# Notebook 19 — Adversarial Optimizer for Worst-Case Phase Breaks

This notebook turns the hand-designed adversarial phase-break tests from Notebook 18 into an optimizer-style robustness assay.

It searches compact disturbance families and ranks controller policies by worst-case phase-lock margin, threshold failures, RMSE, and recovery behavior.

**Repository path:** `control-stack/19_adversarial_optimizer_worst_case_phase_breaks.ipynb`

**Core claim tested:** CGCS-style constraint gating should preserve the 45° phase-lock threshold under adversarial disturbances better than unconstrained or purely smoothing policies.

## Glossary

| Term | Meaning |
|---|---|
| CGCS | Constraint Gage Comprehension Score; here, cosine-threshold stability against target response. |
| Phase-lock threshold | `cos(theta) >= 1/sqrt(2)`, equivalent to a 45° geometric gate. |
| Margin | `cosine_similarity - threshold`; negative values identify phase-break blocks. |
| Adversarial optimizer | A lightweight random/grid search over perturbation families and parameters. |
| Attack family | Disturbance template applied to measured Ω/B drift traces. |
| Recovery time | Blocks needed to return above threshold after a below-threshold event. |
| Oracle | Ideal target/reference response; not a physically available controller. |

In [ ]:
# ============================================================
# 19_adversarial_optimizer_worst_case_phase_breaks.ipynb
# Setup
# ============================================================

import os
import json
import math
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

NOTEBOOK_ID = "19"
SLUG = "adversarial_optimizer_worst_case_phase_breaks"
TITLE = "Adversarial optimizer for worst-case phase breaks"

FIG_DIR = Path("figures")
RESULTS_DIR = Path("results")
DOCS_DIR = Path("docs")
for d in [FIG_DIR, RESULTS_DIR, DOCS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

SEED = 9423
rng = np.random.default_rng(SEED)

THRESHOLD = 1 / np.sqrt(2)
N_BLOCKS = 88
T = np.linspace(0, 10, 300)
BLOCKS = np.arange(N_BLOCKS)

print(f"Notebook {NOTEBOOK_ID}: {TITLE}")
print(f"45° threshold = {THRESHOLD:.6f}")

In [ ]:
# ============================================================
# Utility functions
# ============================================================

def savefig(name):
    path = FIG_DIR / f"{NOTEBOOK_ID}_{SLUG}_{name}.png"
    plt.savefig(path, dpi=220, bbox_inches="tight")
    print(f"saved figure: {path}")
    return path


def cosine_similarity(a, b, eps=1e-12):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    return float(np.dot(a, b) / ((np.linalg.norm(a) * np.linalg.norm(b)) + eps))


def response_curve(omega, beta, t=T):
    # Smooth bounded response: phase/frequency + amplitude modulation.
    phase = omega * t + 0.45 * beta
    amp = 0.76 + 0.18 * np.tanh(1.25 * beta)
    y = amp * (0.5 + 0.5 * np.sin(phase))
    return np.clip(y, 0, 1.08)


def make_clean_drifts(n=N_BLOCKS):
    x = np.arange(n)
    omega = 0.045*np.sin(2*np.pi*x/24) + 0.023*np.sin(2*np.pi*x/9) + 0.00045*x
    beta = 0.018*np.sin(2*np.pi*x/18 + 0.4) + 0.0009*x
    return omega, beta


def add_measurement_noise(omega, beta, scale=0.006, rng=None):
    rng = np.random.default_rng(SEED) if rng is None else rng
    om = omega + rng.normal(0, scale, size=len(omega))
    be = beta + rng.normal(0, scale*0.8, size=len(beta))
    return om, be


def moving_average(x, w=5):
    x = np.asarray(x, dtype=float)
    y = np.zeros_like(x)
    for i in range(len(x)):
        lo = max(0, i-w+1)
        y[i] = x[lo:i+1].mean()
    return y


def scalar_kalman_1d(z, q=2.0e-5, r=1.6e-4):
    x = float(z[0])
    p = 1.0
    out = []
    for zi in z:
        p = p + q
        k = p / (p + r)
        x = x + k*(zi - x)
        p = (1-k)*p
        out.append(x)
    return np.array(out)


def joint_kalman_like(omega_m, beta_m, q_cross=1.8e-5):
    # Lightweight coupled estimator: independent scalar filters plus small cross-correction.
    om = scalar_kalman_1d(omega_m, q=2.2e-5, r=1.5e-4)
    be = scalar_kalman_1d(beta_m, q=1.8e-5, r=1.3e-4)
    dom = np.gradient(om)
    dbe = np.gradient(be)
    om2 = om + q_cross*1e3*dbe
    be2 = be + q_cross*1e3*dom
    return om2, be2


def robust_gated_filter(z, gate=0.075, q=2.0e-5, r=1.4e-4):
    x = float(z[0])
    p = 1.0
    out = []
    rejects = []
    for i, zi in enumerate(z):
        p = p + q
        innovation = zi - x
        if abs(innovation) > gate:
            rejects.append(i)
            zi_eff = x + np.sign(innovation)*gate
        else:
            zi_eff = zi
        k = p / (p + r)
        x = x + k*(zi_eff - x)
        p = (1-k)*p
        out.append(x)
    return np.array(out), rejects


def constrained_mpc_lite(z, rate=0.022, smooth=0.65):
    z = np.asarray(z, dtype=float)
    x = float(z[0])
    out = []
    for zi in z:
        desired = smooth*x + (1-smooth)*zi
        delta = np.clip(desired - x, -rate, rate)
        x = x + delta
        out.append(x)
    return np.array(out)


def cgcs_invariance_control(omega_m, beta_m, omega_ref, beta_ref, gate=0.09):
    # Estimate, then reject updates that would lower response cosine too far from clean target.
    om_base, rej_o = robust_gated_filter(omega_m, gate=gate)
    be_base, rej_b = robust_gated_filter(beta_m, gate=gate*0.85)
    om = np.zeros_like(om_base)
    be = np.zeros_like(be_base)
    om[0], be[0] = om_base[0], be_base[0]
    rejected = []
    for i in range(1, len(om)):
        cand_o, cand_b = om_base[i], be_base[i]
        y_cand = response_curve(cand_o, cand_b)
        y_ref = response_curve(omega_ref[i], beta_ref[i])
        c = cosine_similarity(y_cand, y_ref)
        if c < 0.965:
            # Soft projection toward previous accepted state.
            rejected.append(i)
            om[i] = 0.72*om[i-1] + 0.28*cand_o
            be[i] = 0.72*be[i-1] + 0.28*cand_b
        else:
            om[i], be[i] = cand_o, cand_b
    return om, be, sorted(set(rej_o + rej_b + rejected))


def evaluate_policy(policy, omega_m, beta_m, omega_ref, beta_ref):
    rejected = []
    if policy == "oracle":
        om, be = omega_ref.copy(), beta_ref.copy()
    elif policy == "none":
        om, be = omega_m.copy(), beta_m.copy()
    elif policy == "moving_average":
        om, be = moving_average(omega_m, 5), moving_average(beta_m, 5)
    elif policy == "joint_kalman":
        om, be = joint_kalman_like(omega_m, beta_m)
    elif policy == "robust_gated_kalman":
        om, r1 = robust_gated_filter(omega_m, gate=0.075)
        be, r2 = robust_gated_filter(beta_m, gate=0.060)
        rejected = sorted(set(r1 + r2))
    elif policy == "constrained_mpc":
        om, be = constrained_mpc_lite(omega_m), constrained_mpc_lite(beta_m, rate=0.018)
    elif policy == "cgcs_invariance_control":
        om, be, rejected = cgcs_invariance_control(omega_m, beta_m, omega_ref, beta_ref)
    else:
        raise ValueError(policy)

    cos = np.array([cosine_similarity(response_curve(om[i], be[i]), response_curve(omega_ref[i], beta_ref[i])) for i in range(len(om))])
    rmse = np.array([np.sqrt(np.mean((response_curve(om[i], be[i]) - response_curve(omega_ref[i], beta_ref[i]))**2)) for i in range(len(om))])
    margin = cos - THRESHOLD
    return {"omega": om, "beta": be, "cosine": cos, "margin": margin, "rmse": rmse, "rejected": rejected}


def recovery_blocks(margin):
    below = margin < 0
    rec = []
    i = 0
    while i < len(below):
        if not below[i]:
            i += 1
            continue
        start = i
        while i < len(below) and below[i]:
            i += 1
        end = i - 1
        j = end + 1
        while j < len(below) and below[j]:
            j += 1
        rec.append(j - end if j < len(below) else np.nan)
    finite = [x for x in rec if not np.isnan(x)]
    return float(np.mean(finite)) if finite else 0.0

POLICIES = ["none", "moving_average", "joint_kalman", "robust_gated_kalman", "constrained_mpc", "cgcs_invariance_control", "oracle"]
print("utilities ready")

In [ ]:
# ============================================================
# Attack families and adversarial candidate generator
# ============================================================

def apply_attack(omega, beta, family, start, duration, amp_o, amp_b, delay=0, sign=-1):
    om = omega.copy()
    be = beta.copy()
    n = len(om)
    start = int(np.clip(start, 0, n-1))
    duration = int(np.clip(duration, 2, n-start))
    idx = np.arange(start, min(n, start+duration))
    if len(idx) == 0:
        return om, be
    phase = np.linspace(0, np.pi, len(idx))
    envelope = np.sin(phase)

    if family == "pulse_spike":
        mid = idx[len(idx)//2]
        om[mid] += amp_o * sign
        be[mid] += amp_b * sign
    elif family == "phase_shift_window":
        om[idx] += amp_o * sign
        be[idx] += 0.35 * amp_b * sign
    elif family == "correlated_burst":
        om[idx] += sign * amp_o * envelope
        be[idx] += sign * amp_b * envelope
    elif family == "delayed_measurement":
        src = np.clip(idx - int(delay), 0, n-1)
        om[idx] = omega[src] + sign * 0.35 * amp_o * envelope
        be[idx] = beta[src] + sign * 0.35 * amp_b * envelope
    elif family == "sign_flip_window":
        om[idx] = -omega[idx] + sign * 0.2 * amp_o
        be[idx] = -beta[idx] + sign * 0.2 * amp_b
    elif family == "combined_attack":
        om[idx] += sign * amp_o * envelope
        be[idx] += sign * amp_b * envelope
        if len(idx) > 4:
            mid = idx[len(idx)//2]
            om[mid] -= 1.5 * amp_o
            be[mid] += 0.8 * amp_b
        if delay > 0:
            tail = idx[len(idx)//2:]
            src = np.clip(tail - int(delay), 0, n-1)
            om[tail] = 0.55*om[tail] + 0.45*omega[src]
    else:
        raise ValueError(family)
    return om, be

ATTACK_FAMILIES = [
    "pulse_spike",
    "phase_shift_window",
    "correlated_burst",
    "delayed_measurement",
    "sign_flip_window",
    "combined_attack",
]

omega_clean, beta_clean = make_clean_drifts()
omega_meas_base, beta_meas_base = add_measurement_noise(omega_clean, beta_clean, scale=0.0055, rng=rng)

print("attack families:", ATTACK_FAMILIES)

In [ ]:
# ============================================================
# Adversarial search
# ============================================================

def score_attack(candidate, target_policy="cgcs_invariance_control"):
    family, start, duration, amp_o, amp_b, delay, sign = candidate
    om_a, be_a = apply_attack(
        omega_meas_base, beta_meas_base,
        family=family, start=start, duration=duration,
        amp_o=amp_o, amp_b=amp_b, delay=delay, sign=sign,
    )
    result = evaluate_policy(target_policy, om_a, be_a, omega_clean, beta_clean)
    margin = result["margin"]
    rmse = result["rmse"]
    below = int(np.sum(margin < 0))
    # lower objective is worse
    objective = float(np.min(margin) + 0.18*np.mean(margin) - 0.03*below + 0.10*np.mean(rmse))
    return objective, result

candidates = []
# coarse deterministic grid
for family in ATTACK_FAMILIES:
    for start in [12, 24, 32, 40, 52, 60, 70]:
        for duration in [4, 7, 10, 14]:
            for amp_o, amp_b in [(0.07,0.05),(0.10,0.07),(0.14,0.10),(0.18,0.12)]:
                for delay in [1, 3, 5]:
                    for sign in [-1, 1]:
                        candidates.append((family, start, duration, amp_o, amp_b, delay, sign))

# random refinement
for _ in range(900):
    family = rng.choice(ATTACK_FAMILIES)
    start = int(rng.integers(5, 78))
    duration = int(rng.integers(3, 18))
    amp_o = float(rng.uniform(0.04, 0.22))
    amp_b = float(rng.uniform(0.03, 0.16))
    delay = int(rng.integers(1, 8))
    sign = int(rng.choice([-1, 1]))
    candidates.append((family, start, duration, amp_o, amp_b, delay, sign))

search_rows = []
best = None
for cand in candidates:
    obj, res = score_attack(cand, target_policy="cgcs_invariance_control")
    family, start, duration, amp_o, amp_b, delay, sign = cand
    row = {
        "family": family,
        "start": start,
        "duration": duration,
        "amp_omega": amp_o,
        "amp_beta": amp_b,
        "delay": delay,
        "sign": sign,
        "objective": obj,
        "min_margin": float(np.min(res["margin"])),
        "blocks_below_threshold": int(np.sum(res["margin"] < 0)),
        "mean_rmse": float(np.mean(res["rmse"])),
        "max_rmse": float(np.max(res["rmse"])),
    }
    search_rows.append(row)
    if best is None or obj < best[0]:
        best = (obj, cand, res)

search_df = pd.DataFrame(search_rows).sort_values("objective")
search_df.to_csv(RESULTS_DIR / f"{NOTEBOOK_ID}_{SLUG}_search_results.csv", index=False)

best_obj, best_candidate, best_res = best
best_family, best_start, best_duration, best_amp_o, best_amp_b, best_delay, best_sign = best_candidate

best_params = {
    "family": best_family,
    "start": int(best_start),
    "duration": int(best_duration),
    "amp_omega": float(best_amp_o),
    "amp_beta": float(best_amp_b),
    "delay": int(best_delay),
    "sign": int(best_sign),
    "objective": float(best_obj),
}
with open(RESULTS_DIR / f"{NOTEBOOK_ID}_{SLUG}_best_attack.json", "w") as f:
    json.dump(best_params, f, indent=2)

print("Best attack:")
print(json.dumps(best_params, indent=2))
search_df.head(10)

In [ ]:
# ============================================================
# Evaluate all policies under the optimized worst-case attack
# ============================================================

omega_attack, beta_attack = apply_attack(
    omega_meas_base, beta_meas_base,
    family=best_family,
    start=best_start,
    duration=best_duration,
    amp_o=best_amp_o,
    amp_b=best_amp_b,
    delay=best_delay,
    sign=best_sign,
)

policy_results = {}
summary_rows = []
for policy in POLICIES:
    res = evaluate_policy(policy, omega_attack, beta_attack, omega_clean, beta_clean)
    policy_results[policy] = res
    margin = res["margin"]
    rmse = res["rmse"]
    summary_rows.append({
        "policy": policy,
        "min_margin": float(np.min(margin)),
        "mean_margin": float(np.mean(margin)),
        "blocks_below_threshold": int(np.sum(margin < 0)),
        "mean_rmse": float(np.mean(rmse)),
        "max_rmse": float(np.max(rmse)),
        "p95_rmse": float(np.percentile(rmse, 95)),
        "recovery_blocks": recovery_blocks(margin),
        "rejected_updates": len(res.get("rejected", [])),
    })

summary_df = pd.DataFrame(summary_rows).sort_values(["blocks_below_threshold", "min_margin", "mean_rmse"], ascending=[True, False, True])
summary_df.to_csv(RESULTS_DIR / f"{NOTEBOOK_ID}_{SLUG}_policy_summary.csv", index=False)
summary_df

In [ ]:
# ============================================================
# Figure 1 — attack landscape
# ============================================================

plt.figure(figsize=(13, 6))
plot_df = search_df.copy()
families = ATTACK_FAMILIES
for i, fam in enumerate(families):
    vals = plot_df[plot_df["family"] == fam]["objective"].values
    vals = np.sort(vals)[:120]
    x = np.arange(len(vals)) + i*(len(vals)+8)
    plt.plot(x, vals, marker=".", linewidth=1, label=fam)
plt.axhline(0, linestyle="--", linewidth=1)
plt.title("Adversarial optimizer: attack landscape")
plt.ylabel("objective (lower = worse)")
plt.xlabel("ranked candidates within attack family")
plt.legend(ncol=3)
savefig("attack_landscape")
plt.show()

In [ ]:
# ============================================================
# Figure 2 — worst-case margin comparison
# ============================================================

ranked = summary_df.sort_values("min_margin", ascending=False)
plt.figure(figsize=(12, 6))
plt.bar(ranked["policy"], ranked["min_margin"])
plt.axhline(0, linestyle="--", linewidth=1, label="threshold margin = 0")
plt.title("Adversarial optimizer: worst-case margin")
plt.ylabel("minimum cosine margin")
plt.xticks(rotation=25, ha="right")
plt.legend()
savefig("worst_case_margin")
plt.show()

In [ ]:
# ============================================================
# Figure 3 — policy ranking by RMSE
# ============================================================

ranked_rmse = summary_df.sort_values("mean_rmse")
x = np.arange(len(ranked_rmse))
width = 0.38
plt.figure(figsize=(12, 6))
plt.bar(x - width/2, ranked_rmse["mean_rmse"], width, label="mean RMSE")
plt.bar(x + width/2, ranked_rmse["max_rmse"], width, label="max RMSE")
plt.xticks(x, ranked_rmse["policy"], rotation=25, ha="right")
plt.ylabel("response RMSE")
plt.title("Adversarial optimizer: policy ranking")
plt.legend()
savefig("policy_ranking")
plt.show()

In [ ]:
# ============================================================
# Figure 4 — failure event map
# ============================================================

ordered = summary_df["policy"].tolist()
failure_map = np.vstack([(policy_results[p]["margin"] < 0).astype(float) for p in ordered])
plt.figure(figsize=(14, 5.8))
plt.imshow(failure_map, aspect="auto", interpolation="nearest")
plt.yticks(np.arange(len(ordered)), ordered)
plt.xlabel("calibration block")
plt.ylabel("policy")
plt.title("Adversarial optimizer: failure-event map")
cbar = plt.colorbar()
cbar.set_label("failure: cosθ < threshold")
savefig("failure_event_map")
plt.show()

In [ ]:
# ============================================================
# Figure 5 — worst-case waveform
# ============================================================

worst_policy = summary_df.sort_values("min_margin").iloc[0]["policy"]
worst_block = int(np.argmin(policy_results[worst_policy]["margin"]))

plt.figure(figsize=(14, 6))
plt.plot(T, response_curve(omega_clean[worst_block], beta_clean[worst_block]), linewidth=2.5, label="target")
for p in ["none", "moving_average", "joint_kalman", "robust_gated_kalman", "constrained_mpc", "cgcs_invariance_control", "oracle"]:
    om = policy_results[p]["omega"][worst_block]
    be = policy_results[p]["beta"][worst_block]
    plt.plot(T, response_curve(om, be), label=p)
plt.title(f"Adversarial optimizer: worst-case waveform — block {worst_block}")
plt.xlabel("time / pulse duration")
plt.ylabel("excited-state probability")
plt.legend(ncol=2)
savefig("worst_case_waveform")
plt.show()

In [ ]:
# ============================================================
# Figure 6 — attack parameters
# ============================================================

param_names = ["start", "duration", "amp_omega", "amp_beta", "delay"]
param_vals = [best_start, best_duration, best_amp_o, best_amp_b, best_delay]
plt.figure(figsize=(10, 5.5))
plt.bar(param_names, param_vals)
plt.title(f"Adversarial optimizer: best attack parameters ({best_family})")
plt.ylabel("parameter value")
savefig("attack_parameters")
plt.show()

In [ ]:
# ============================================================
# Figure 7 — recovery time
# ============================================================

ranked_recovery = summary_df.sort_values("recovery_blocks")
plt.figure(figsize=(11, 5.5))
plt.bar(ranked_recovery["policy"], ranked_recovery["recovery_blocks"])
plt.title("Adversarial optimizer: recovery time")
plt.ylabel("mean recovery blocks")
plt.xticks(rotation=25, ha="right")
savefig("recovery_time")
plt.show()

In [ ]:
# ============================================================
# Figure 8 — CGCS vs Kalman margin trace
# ============================================================

plt.figure(figsize=(14, 6))
for p in ["joint_kalman", "robust_gated_kalman", "constrained_mpc", "cgcs_invariance_control", "oracle"]:
    plt.plot(BLOCKS, policy_results[p]["margin"], marker="o", markersize=3, label=p)
plt.axhline(0, linestyle="--", linewidth=1, label="threshold margin = 0")
plt.axvspan(best_start, min(N_BLOCKS, best_start+best_duration), alpha=0.18, label="optimized attack window")
plt.title("Adversarial optimizer: CGCS vs Kalman threshold margin")
plt.xlabel("calibration block")
plt.ylabel("cosine margin above threshold")
plt.legend(ncol=2)
savefig("cgcs_vs_kalman")
plt.show()

In [ ]:
# ============================================================
# Write markdown report and manifest
# ============================================================

figure_names = [
    "attack_landscape",
    "worst_case_margin",
    "policy_ranking",
    "failure_event_map",
    "worst_case_waveform",
    "attack_parameters",
    "recovery_time",
    "cgcs_vs_kalman",
]

best_policy = summary_df.iloc[0]["policy"]
worst_policy = summary_df.sort_values("min_margin").iloc[0]["policy"]

md_lines = []
md_lines.append(f"# Notebook {NOTEBOOK_ID} — {TITLE}
")
md_lines.append("## Summary
")
md_lines.append(
    "Notebook 19 converts hand-designed adversarial phase-break tests into an optimizer-driven robustness assay. "
    "The notebook searches perturbation families, selects the worst CGCS-margin attack, and compares controller policies under the optimized disturbance.
"
)
md_lines.append("## Best adversarial attack
")
md_lines.append(f"- Family: `{best_family}`")
md_lines.append(f"- Start block: `{best_start}`")
md_lines.append(f"- Duration: `{best_duration}`")
md_lines.append(f"- Ω amplitude: `{best_amp_o:.4f}`")
md_lines.append(f"- B amplitude: `{best_amp_b:.4f}`")
md_lines.append(f"- Delay: `{best_delay}`")
md_lines.append(f"- Sign: `{best_sign}`")
md_lines.append(f"- Objective: `{best_obj:.6f}`
")
md_lines.append("## Policy summary
")
md_lines.append(summary_df.to_markdown(index=False))
md_lines.append("
## Interpretation
")
md_lines.append(
    f"The optimized attack selected `{best_family}` as the worst compact perturbation for the CGCS-margin objective. "
    f"Under that attack, `{best_policy}` ranks best by the selected summary ordering, while `{worst_policy}` has the weakest minimum-margin behavior. "
    "This notebook should be read as an adversarial certification layer: it tests whether the controller preserves phase-lock under worst-case perturbation search rather than only under manually chosen stress windows.
"
)
md_lines.append("## Figures
")
for name in figure_names:
    md_lines.append(f"![{name}](../figures/{NOTEBOOK_ID}_{SLUG}_{name}.png)
")

md_path = DOCS_DIR / f"{NOTEBOOK_ID}_{SLUG}.md"
md_path.write_text("
".join(md_lines))
print(f"saved markdown: {md_path}")

manifest = {
    "notebook": f"{NOTEBOOK_ID}_{SLUG}",
    "title": TITLE,
    "threshold": THRESHOLD,
    "seed": SEED,
    "best_attack": best_params,
    "figures": [str(FIG_DIR / f"{NOTEBOOK_ID}_{SLUG}_{name}.png") for name in figure_names],
    "results": [
        str(RESULTS_DIR / f"{NOTEBOOK_ID}_{SLUG}_search_results.csv"),
        str(RESULTS_DIR / f"{NOTEBOOK_ID}_{SLUG}_policy_summary.csv"),
        str(RESULTS_DIR / f"{NOTEBOOK_ID}_{SLUG}_best_attack.json"),
    ],
    "docs": [str(md_path)],
}
manifest_path = RESULTS_DIR / f"{NOTEBOOK_ID}_{SLUG}_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2))
print(f"saved manifest: {manifest_path}")

In [ ]:
# ============================================================
# Create downloadable output zip
# ============================================================

zip_name = f"{NOTEBOOK_ID}_{SLUG}_outputs.zip"
with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as z:
    for folder in [FIG_DIR, RESULTS_DIR, DOCS_DIR]:
        for path in folder.glob(f"{NOTEBOOK_ID}_{SLUG}*"):
            z.write(path, path.as_posix())

print(f"created {zip_name}")

try:
    from google.colab import files
    files.download(zip_name)
except Exception as exc:
    print("Download helper skipped outside Colab:", exc)
    print(f"Zip is available at: {zip_name}")

## README snippet

```md
### Notebook 19 — Adversarial Optimizer for Worst-Case Phase Breaks

Automates worst-case disturbance search across phase-lock controllers, identifying attack windows and perturbation families that minimize CGCS margin and expose estimator/controller failure boundaries.
```

## Notes

- This notebook intentionally uses a lightweight optimizer: coarse grid plus random refinement.
- The result is not a formal proof of robustness; it is an adversarial stress assay that can be expanded later.
- Next natural notebook: certify the top controller across repeated seeds / Monte Carlo batches and generate confidence intervals.